# 01 — Unconditional Generation

Generate monomer backbones at lengths {100, 150, 250} with both models, then measure:

- wall-clock per design
- structural diversity (pairwise approx-TM)
- secondary-structure profile

In [ ]:
%cd /content/repo
import sys
if '/content/repo/scripts' not in sys.path:
    sys.path.insert(0, '/content/repo/scripts')

from utils import RESULTS, DATA, RunRecord, append_record, free_gpu, rfd3_run
import numpy as np, json, time, os
from pathlib import Path

LENGTHS = [100, 150, 250]
N_PER_LEN = 8         # bump to 32 on A100

## Chroma — unconditional sampling

In [ ]:
from chroma import Chroma, api
api.register_key(os.environ['CHROMA_API_KEY'])
chroma = Chroma()

chroma_out = DATA / 'chroma_uncond'
chroma_out.mkdir(exist_ok=True)

for L in LENGTHS:
    times = []
    for i in range(N_PER_LEN):
        t0 = time.perf_counter()
        protein = chroma.sample(chain_lengths=[L], steps=200, sde_func='langevin')
        times.append(time.perf_counter() - t0)
        protein.to(str(chroma_out / f'L{L}_n{i:02d}.pdb'))
    append_record(RunRecord(
        model='chroma', task='uncond', target='-',
        length=L, n_designs=N_PER_LEN, seconds=sum(times),
        metrics={'s_per_design': float(np.mean(times))},
    ))
    print(f'Chroma L={L}: {np.mean(times):.1f}s/design')

free_gpu()

## RFdiffusion3 — unconditional sampling

RFD3 takes a JSON spec and `diffusion_batch_size` controls designs/batch.

In [ ]:
rfd3_out = DATA / 'rfd3_uncond'
rfd3_out.mkdir(exist_ok=True)

for L in LENGTHS:
    spec = {f'uncond_L{L}': {'length': f'{L}-{L}'}}
    out_dir = rfd3_out / f'L{L}'
    ok, err, dt = rfd3_run(spec, out_dir,
                            diffusion_batch_size=N_PER_LEN,
                            n_batches=1,
                            num_timesteps=200)
    if not ok:
        print(f'RFD3 L={L}: FAILED — {err[-200:]}')
        continue
    append_record(RunRecord(
        model='rfd3', task='uncond', target='-',
        length=L, n_designs=N_PER_LEN, seconds=dt,
        metrics={'s_per_design': dt / N_PER_LEN},
    ))
    print(f'RFD3 L={L}: {dt/N_PER_LEN:.1f}s/design')

free_gpu()

## Diversity (pairwise approx-TM)

In [ ]:
from utils import load_ca_coords, tm_score
import itertools

def find_designs(root, L, cap=N_PER_LEN):
    files = sorted(root.rglob(f'*L{L}*.pdb'))[:cap]
    return [f for f in files if f.is_file()]

def pairwise_tm(pdb_files):
    coords = [load_ca_coords(p) for p in pdb_files]
    coords = [c for c in coords if len(c) >= 30]
    n = len(coords)
    if n < 2: return None
    M = np.zeros((n, n))
    for i, j in itertools.combinations(range(n), 2):
        M[i, j] = M[j, i] = tm_score(coords[i], coords[j])
    return M

def diversity_score(M):
    if M is None: return 0.0
    iu = np.triu_indices_from(M, k=1)
    return float(1 - M[iu].mean())

div_results = {}
for model, root in [('chroma', chroma_out), ('rfd3', rfd3_out)]:
    for L in LENGTHS:
        files = find_designs(root, L)
        if len(files) < 2:
            print(f'{model} L={L}: skip (only {len(files)} files)')
            continue
        M = pairwise_tm(files)
        div_results[(model, L)] = {'diversity': diversity_score(M), 'n': len(files)}
        print(f'{model:7s} L={L}: diversity={div_results[(model,L)]["diversity"]:.3f}  '
              f'(n={len(files)})')

In [ ]:
# Secondary-structure profile (Biotite annotate_sse needs full backbone)
import biotite.structure.io.pdb as bpdb
import biotite.structure as bstruc

def ss_fractions(pdb_path):
    try:
        arr = bpdb.PDBFile.read(pdb_path).get_structure(model=1)
    except Exception:
        return 0., 0., 1.
    chains = np.unique(arr.chain_id)
    arr = arr[arr.chain_id == chains[0]]
    try:
        sse = bstruc.annotate_sse(arr)
        return (float((sse == 'a').mean()),
                float((sse == 'b').mean()),
                float((sse == 'c').mean()))
    except Exception:
        return 0., 0., 1.

ss_results = {}
for model, root in [('chroma', chroma_out), ('rfd3', rfd3_out)]:
    for L in LENGTHS:
        files = find_designs(root, L)
        if not files: continue
        fracs = np.array([ss_fractions(p) for p in files])
        h, e, c = fracs.mean(0)
        ss_results[(model, L)] = {'helix': float(h), 'sheet': float(e),
                                  'coil':  float(c), 'n': len(files)}
        print(f'{model:7s} L={L}: H={h:.2f}  E={e:.2f}  C={c:.2f}')

## Plots

In [ ]:
import matplotlib.pyplot as plt
from utils import load_records

recs = [r for r in load_records() if r['task'] == 'uncond']

fig, ax = plt.subplots(figsize=(7, 4))
for model, mk, c in [('chroma', 'o', 'tab:blue'), ('rfd3', 's', 'tab:orange')]:
    pts = [(r['length'], r['metrics']['s_per_design']) for r in recs if r['model'] == model]
    if pts:
        xs, ys = zip(*sorted(pts))
        ax.plot(xs, ys, marker=mk, color=c, label=model.upper(), lw=2)
ax.set_xlabel('Length (residues)')
ax.set_ylabel('Seconds / design')
ax.set_title('Wall-clock per design')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'fig_uncond_efficiency.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
x = np.arange(len(LENGTHS))
for i, model in enumerate(['chroma', 'rfd3']):
    ys = [div_results.get((model, L), {}).get('diversity', 0) for L in LENGTHS]
    ax.bar(x + i*width, ys, width, label=model.upper())
ax.set_xticks(x + width/2); ax.set_xticklabels(LENGTHS)
ax.set_xlabel('Length')
ax.set_ylabel('1 − mean pairwise TM (higher = more diverse)')
ax.set_title('Structural diversity')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_uncond_diversity.png', dpi=150)
plt.show()

In [ ]:
out = {
    'efficiency': recs,
    'diversity': {f'{m}_L{L}': v for (m, L), v in div_results.items()},
    'ss_fractions': {f'{m}_L{L}': v for (m, L), v in ss_results.items()},
}
(RESULTS / 'uncond_summary.json').write_text(json.dumps(out, indent=2, default=str))
print('Saved uncond_summary.json')